In [0]:
# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ============================================================
# PROJECT CONFIGURATION
# ============================================================

CATALOG = "aml_engine"
SCHEMA = "aml_poc"

SILVER_TX_TABLE = (
    f"{CATALOG}.{SCHEMA}.silver_transactions"
)

SILVER_ACCOUNTS_TABLE = (
    f"{CATALOG}.{SCHEMA}.silver_accounts"
)

GRAPH_RESULTS_TABLE = (
    f"{CATALOG}.{SCHEMA}.graph_results"
)

ML_TRAINING_TABLE = (
    f"{CATALOG}.{SCHEMA}.ml_training_data"
)


# ============================================================
# COMMON S3 LOCATION
# ============================================================

S3_BASE_PATH = (
    "s3://zubair-s3-demo/raw_dataset/aml"
)

S3_DELTA_PATH = (
    f"{S3_BASE_PATH}/delta_tables"
)

ML_TRAINING_PATH = (
    f"{S3_DELTA_PATH}/ml_training_data"
)


# ============================================================
# FEATURE PARAMETERS
# ============================================================

# Number of event-time units used for recent activity.
LOOKBACK_WINDOW = 5

# Current POC rule thresholds.
HIGH_VALUE_THRESHOLD = 1000.0

VELOCITY_THRESHOLD = 5

FAN_IN_THRESHOLD = 5

FAN_OUT_THRESHOLD = 5


print("ML training-data configuration loaded.")
print("Training table :", ML_TRAINING_TABLE)
print("S3 location    :", ML_TRAINING_PATH)

In [0]:
# COMMAND ----------

transactions = (
    spark.table(SILVER_TX_TABLE)
    .select(
        "tx_id",
        "sender_account_id",
        "receiver_account_id",
        "tx_type",
        "tx_amount",
        "event_time",
        "is_fraud"
    )
)


print(
    "Transaction count:",
    transactions.count()
)

print(
    "Transaction columns:",
    transactions.columns
)

In [0]:
# COMMAND ----------

display(
    transactions.limit(10)
)

In [0]:
# COMMAND ----------

display(
    transactions
    .groupBy("is_fraud")
    .count()
    .orderBy("is_fraud")
)

In [0]:
# COMMAND ----------

ml_df = (
    transactions
    .withColumn(
        "label",
        F.col("is_fraud").cast("int")
    )
)

In [0]:
# COMMAND ----------

display(
    ml_df
    .select(
        "tx_id",
        "is_fraud",
        "label"
    )
    .limit(20)
)

In [0]:
# COMMAND ----------

accounts = spark.table(
    SILVER_ACCOUNTS_TABLE
)

print(
    "Account count:",
    accounts.count()
)

print(
    "Account columns:",
    accounts.columns
)

In [0]:
# COMMAND ----------

sender_accounts = (
    accounts
    .select(
        F.col("account_id")
            .alias("sender_account_id"),

        F.col("country")
            .alias("sender_country"),

        F.col("account_type")
            .alias("sender_account_type"),

        F.col("init_balance")
            .alias("sender_init_balance")
    )
)

In [0]:
# COMMAND ----------

receiver_accounts = (
    accounts
    .select(
        F.col("account_id")
            .alias("receiver_account_id"),

        F.col("country")
            .alias("receiver_country"),

        F.col("account_type")
            .alias("receiver_account_type"),

        F.col("init_balance")
            .alias("receiver_init_balance")
    )
)

In [0]:
# COMMAND ----------

ml_df = (
    ml_df

    .join(
        sender_accounts,
        on="sender_account_id",
        how="left"
    )

    .join(
        receiver_accounts,
        on="receiver_account_id",
        how="left"
    )
)

In [0]:
# COMMAND ----------

sender_history_window = (
    Window
    .partitionBy("sender_account_id")
    .orderBy("event_time")
    .rangeBetween(
        Window.unboundedPreceding,
        -1
    )
)

In [0]:
# COMMAND ----------

ml_df = (
    ml_df

    .withColumn(
        "sender_tx_count_before",
        F.count("tx_id")
        .over(sender_history_window)
    )

    .withColumn(
        "sender_total_amount_before",
        F.coalesce(
            F.sum("tx_amount")
            .over(sender_history_window),
            F.lit(0.0)
        )
    )
)

In [0]:
# COMMAND ----------

receiver_history_window = (
    Window
    .partitionBy("receiver_account_id")
    .orderBy("event_time")
    .rangeBetween(
        Window.unboundedPreceding,
        -1
    )
)

In [0]:
# COMMAND ----------

ml_df = (
    ml_df

    .withColumn(
        "receiver_tx_count_before",
        F.count("tx_id")
        .over(receiver_history_window)
    )

    .withColumn(
        "receiver_total_amount_before",
        F.coalesce(
            F.sum("tx_amount")
            .over(receiver_history_window),
            F.lit(0.0)
        )
    )
)

In [0]:
# COMMAND ----------

sender_velocity_window = (
    Window
    .partitionBy("sender_account_id")
    .orderBy("event_time")
    .rangeBetween(
        -LOOKBACK_WINDOW,
        -1
    )
)

In [0]:
# COMMAND ----------

ml_df = (
    ml_df
    .withColumn(
        "sender_velocity_count",
        F.count("tx_id")
        .over(sender_velocity_window)
    )
)

In [0]:
# COMMAND ----------

receiver_velocity_window = (
    Window
    .partitionBy("receiver_account_id")
    .orderBy("event_time")
    .rangeBetween(
        -LOOKBACK_WINDOW,
        -1
    )
)

ml_df = (
    ml_df
    .withColumn(
        "receiver_velocity_count",
        F.count("tx_id")
        .over(receiver_velocity_window)
    )
)

In [0]:
# COMMAND ----------

sender_receiver_first_seen = (
    transactions
    .groupBy(
        "sender_account_id",
        "receiver_account_id"
    )
    .agg(
        F.min("event_time")
        .alias("first_seen_time")
    )
)

In [0]:
# COMMAND ----------

new_receiver_counts = (
    sender_receiver_first_seen
    .groupBy(
        "sender_account_id",
        "first_seen_time"
    )
    .agg(
        F.count("*")
        .alias("new_receivers")
    )
)

In [0]:
# COMMAND ----------

unique_receiver_history_window = (
    Window
    .partitionBy("sender_account_id")
    .orderBy("first_seen_time")
    .rangeBetween(
        Window.unboundedPreceding,
        -1
    )
)

new_receiver_counts = (
    new_receiver_counts
    .withColumn(
        "unique_receivers_before",
        F.coalesce(
            F.sum("new_receivers")
            .over(unique_receiver_history_window),
            F.lit(0)
        )
    )
)

In [0]:
# COMMAND ----------

receiver_lookup = (
    new_receiver_counts
    .select(
        "sender_account_id",

        F.col("first_seen_time")
        .alias("relationship_time"),

        "unique_receivers_before"
    )
)

ml_df = (
    ml_df
    .join(
        receiver_lookup,
        (
            (ml_df.sender_account_id ==
             receiver_lookup.sender_account_id)
            &
            (ml_df.event_time ==
             receiver_lookup.relationship_time)
        ),
        "left"
    )
    .drop(
        receiver_lookup.sender_account_id,
        "relationship_time"
    )
    .fillna(
        {
            "unique_receivers_before": 0
        }
    )
)

In [0]:
# COMMAND ----------

receiver_sender_first_seen = (
    transactions
    .groupBy(
        "receiver_account_id",
        "sender_account_id"
    )
    .agg(
        F.min("event_time")
        .alias("first_seen_time")
    )
)

In [0]:
# COMMAND ----------

new_sender_counts = (
    receiver_sender_first_seen
    .groupBy(
        "receiver_account_id",
        "first_seen_time"
    )
    .agg(
        F.count("*")
        .alias("new_senders")
    )
)

In [0]:
# COMMAND ----------

unique_sender_history_window = (
    Window
    .partitionBy("receiver_account_id")
    .orderBy("first_seen_time")
    .rangeBetween(
        Window.unboundedPreceding,
        -1
    )
)

new_sender_counts = (
    new_sender_counts
    .withColumn(
        "unique_senders_before",
        F.coalesce(
            F.sum("new_senders")
            .over(unique_sender_history_window),
            F.lit(0)
        )
    )
)

In [0]:
# COMMAND ----------

sender_lookup = (
    new_sender_counts
    .select(
        "receiver_account_id",

        F.col("first_seen_time")
        .alias("relationship_time"),

        "unique_senders_before"
    )
)

ml_df = (
    ml_df
    .join(
        sender_lookup,
        (
            (ml_df.receiver_account_id ==
             sender_lookup.receiver_account_id)
            &
            (ml_df.event_time ==
             sender_lookup.relationship_time)
        ),
        "left"
    )
    .drop(
        sender_lookup.receiver_account_id,
        "relationship_time"
    )
    .fillna(
        {
            "unique_senders_before": 0
        }
    )
)

In [0]:
# COMMAND ----------

ml_df = (
    ml_df
    .withColumn(
        "high_value_flag",
        F.when(
            F.col("tx_amount")
            > HIGH_VALUE_THRESHOLD,
            1
        ).otherwise(0)
    )
)

In [0]:
# COMMAND ----------

ml_df = (
    ml_df
    .withColumn(
        "velocity_flag",
        F.when(
            F.col("sender_velocity_count")
            >= VELOCITY_THRESHOLD,
            1
        ).otherwise(0)
    )
)

In [0]:
# COMMAND ----------

ml_df = (
    ml_df
    .withColumn(
        "fan_in_feature",
        F.col("unique_senders_before")
    )
)

In [0]:
# COMMAND ----------

ml_df = (
    ml_df
    .withColumn(
        "fan_in_flag",
        F.when(
            F.col("fan_in_feature")
            >= FAN_IN_THRESHOLD,
            1
        ).otherwise(0)
    )
)

In [0]:
# COMMAND ----------

ml_df = (
    ml_df
    .withColumn(
        "fan_out_feature",
        F.col("unique_receivers_before")
    )
)

In [0]:
# COMMAND ----------

ml_df = (
    ml_df
    .withColumn(
        "fan_out_flag",
        F.when(
            F.col("fan_out_feature")
            >= FAN_OUT_THRESHOLD,
            1
        ).otherwise(0)
    )
)

In [0]:
# COMMAND ----------

sender_edges = (
    transactions
    .select(
        "sender_account_id",
        "receiver_account_id",
        "event_time"
    )
    .dropDuplicates()
)

In [0]:
# COMMAND ----------

sender_edge_first_seen = (
    sender_edges
    .groupBy(
        "sender_account_id",
        "receiver_account_id"
    )
    .agg(
        F.min("event_time")
        .alias("first_seen_time")
    )
)

In [0]:
# COMMAND ----------

sender_new_edges = (
    sender_edge_first_seen
    .groupBy(
        "sender_account_id",
        "first_seen_time"
    )
    .agg(
        F.count("*")
        .alias("new_outgoing_edges")
    )
)

In [0]:
# COMMAND ----------

sender_degree_window = (
    Window
    .partitionBy("sender_account_id")
    .orderBy("first_seen_time")
    .rangeBetween(
        Window.unboundedPreceding,
        -1
    )
)

sender_new_edges = (
    sender_new_edges
    .withColumn(
        "sender_out_degree_before",
        F.coalesce(
            F.sum("new_outgoing_edges")
            .over(sender_degree_window),
            F.lit(0)
        )
    )
)

In [0]:
# COMMAND ----------

sender_degree_lookup = (
    sender_new_edges
    .select(
        "sender_account_id",

        F.col("first_seen_time")
        .alias("degree_event_time"),

        "sender_out_degree_before"
    )
)

ml_df = (
    ml_df
    .join(
        sender_degree_lookup,
        (
            (ml_df.sender_account_id ==
             sender_degree_lookup.sender_account_id)
            &
            (ml_df.event_time ==
             sender_degree_lookup.degree_event_time)
        ),
        "left"
    )
    .drop(
        sender_degree_lookup.sender_account_id,
        "degree_event_time"
    )
    .fillna(
        {
            "sender_out_degree_before": 0
        }
    )
)

In [0]:
# COMMAND ----------

receiver_edges = (
    transactions
    .select(
        "receiver_account_id",
        "sender_account_id",
        "event_time"
    )
    .dropDuplicates()
)

In [0]:
# COMMAND ----------

receiver_edge_first_seen = (
    receiver_edges
    .groupBy(
        "receiver_account_id",
        "sender_account_id"
    )
    .agg(
        F.min("event_time")
        .alias("first_seen_time")
    )
)

In [0]:
# COMMAND ----------

receiver_new_edges = (
    receiver_edge_first_seen
    .groupBy(
        "receiver_account_id",
        "first_seen_time"
    )
    .agg(
        F.count("*")
        .alias("new_incoming_edges")
    )
)

In [0]:
# COMMAND ----------

receiver_degree_window = (
    Window
    .partitionBy("receiver_account_id")
    .orderBy("first_seen_time")
    .rangeBetween(
        Window.unboundedPreceding,
        -1
    )
)

receiver_new_edges = (
    receiver_new_edges
    .withColumn(
        "receiver_in_degree_before",
        F.coalesce(
            F.sum("new_incoming_edges")
            .over(receiver_degree_window),
            F.lit(0)
        )
    )
)

In [0]:
# COMMAND ----------

receiver_degree_lookup = (
    receiver_new_edges
    .select(
        "receiver_account_id",

        F.col("first_seen_time")
        .alias("degree_event_time"),

        "receiver_in_degree_before"
    )
)

ml_df = (
    ml_df
    .join(
        receiver_degree_lookup,
        (
            (ml_df.receiver_account_id ==
             receiver_degree_lookup.receiver_account_id)
            &
            (ml_df.event_time ==
             receiver_degree_lookup.degree_event_time)
        ),
        "left"
    )
    .drop(
        receiver_degree_lookup.receiver_account_id,
        "degree_event_time"
    )
    .fillna(
        {
            "receiver_in_degree_before": 0
        }
    )
)

In [0]:
# COMMAND ----------

graph_results = spark.table(
    GRAPH_RESULTS_TABLE
)

In [0]:
# COMMAND ----------

cycle_times = (
    graph_results
    .select(
        "cycle_id",
        "tx_id"
    )
    .dropDuplicates()
    .join(
        transactions.select(
            "tx_id",
            "event_time"
        ),
        on="tx_id",
        how="inner"
    )
)

In [0]:
# COMMAND ----------

cycle_completion = (
    cycle_times
    .groupBy("cycle_id")
    .agg(
        F.max("event_time")
        .alias("cycle_completion_time")
    )
)

In [0]:
# COMMAND ----------

cycle_transaction_features = (
    cycle_times
    .join(
        cycle_completion,
        on="cycle_id",
        how="left"
    )
)

In [0]:
# COMMAND ----------

cycle_features = (
    cycle_transaction_features
    .groupBy("tx_id")
    .agg(
        F.max(
            F.when(
                F.col("event_time")
                >= F.col("cycle_completion_time"),
                1
            ).otherwise(0)
        ).alias("cycle_detected_as_of_time"),

        F.countDistinct(
            "cycle_id"
        ).alias("cycle_count")
    )
)

In [0]:
# COMMAND ----------

ml_df = (
    ml_df
    .join(
        cycle_features,
        on="tx_id",
        how="left"
    )
    .fillna(
        {
            "cycle_detected_as_of_time": 0,
            "cycle_count": 0
        }
    )
)

In [0]:
# COMMAND ----------

numeric_features = [
    "sender_tx_count_before",
    "receiver_tx_count_before",

    "sender_total_amount_before",
    "receiver_total_amount_before",

    "sender_velocity_count",
    "receiver_velocity_count",

    "unique_receivers_before",
    "unique_senders_before",

    "fan_in_feature",
    "fan_out_feature",

    "high_value_flag",
    "velocity_flag",

    "fan_in_flag",
    "fan_out_flag",

    "sender_out_degree_before",
    "receiver_in_degree_before",

    "cycle_detected_as_of_time",
    "cycle_count"
]

ml_df = (
    ml_df
    .fillna(
        0,
        subset=numeric_features
    )
)

In [0]:
# COMMAND ----------

ml_training_data = ml_df.select(

    # ========================================================
    # Identifier
    # ========================================================

    "tx_id",

    # ========================================================
    # TRANSACTION FEATURES
    # ========================================================

    "tx_amount",
    "tx_type",
    "event_time",

    # ========================================================
    # ACCOUNT FEATURES
    # ========================================================

    "sender_country",
    "receiver_country",

    "sender_account_type",
    "receiver_account_type",

    "sender_init_balance",
    "receiver_init_balance",

    # ========================================================
    # HISTORICAL BEHAVIOR
    # ========================================================

    "sender_tx_count_before",
    "receiver_tx_count_before",

    "sender_total_amount_before",
    "receiver_total_amount_before",

    "sender_velocity_count",
    "receiver_velocity_count",

    "unique_receivers_before",
    "unique_senders_before",

    # ========================================================
    # RULE FLAGS
    # ========================================================

    "high_value_flag",
    "velocity_flag",

    "fan_in_flag",
    "fan_out_flag",

    # ========================================================
    # RULE BEHAVIOR COUNTS
    # ========================================================

    "fan_in_feature",
    "fan_out_feature",

    # ========================================================
    # TIME-AWARE GRAPH FEATURES
    # ========================================================

    "sender_out_degree_before",
    "receiver_in_degree_before",

    "cycle_detected_as_of_time",
    "cycle_count",

    # ========================================================
    # TARGET
    # ========================================================

    "label"
)

In [0]:
# COMMAND ----------

feature_columns = [
    "tx_amount",
    "tx_type",
    "event_time",

    "sender_country",
    "receiver_country",

    "sender_account_type",
    "receiver_account_type",

    "sender_init_balance",
    "receiver_init_balance",

    "sender_tx_count_before",
    "receiver_tx_count_before",

    "sender_total_amount_before",
    "receiver_total_amount_before",

    "sender_velocity_count",
    "receiver_velocity_count",

    "unique_receivers_before",
    "unique_senders_before",

    "high_value_flag",
    "velocity_flag",

    "fan_in_flag",
    "fan_out_flag",

    "fan_in_feature",
    "fan_out_feature",

    "sender_out_degree_before",
    "receiver_in_degree_before",

    "cycle_detected_as_of_time",
    "cycle_count"
]

target_column = "label"

print("Number of ML features:", len(feature_columns))
print("Target:", target_column)

In [0]:
# COMMAND ----------

transaction_count = transactions.count()

training_count = ml_training_data.count()

print("Silver transactions:", transaction_count)
print("ML training records:", training_count)

In [0]:
# COMMAND ----------

unique_training_transactions = (
    ml_training_data
    .select("tx_id")
    .distinct()
    .count()
)

print(
    "Unique training transactions:",
    unique_training_transactions
)

In [0]:
# COMMAND ----------

display(
    ml_training_data
    .groupBy("label")
    .count()
    .orderBy("label")
)

In [0]:
# COMMAND ----------

null_check = ml_training_data.select(
    [
        F.sum(
            F.when(
                F.col(c).isNull(),
                1
            ).otherwise(0)
        ).alias(c)
        for c in ml_training_data.columns
    ]
)

display(null_check)

In [0]:
# COMMAND ----------

display(
    ml_training_data.select(
        "tx_amount",

        "sender_tx_count_before",
        "receiver_tx_count_before",

        "sender_total_amount_before",
        "receiver_total_amount_before",

        "sender_velocity_count",
        "receiver_velocity_count",

        "unique_receivers_before",
        "unique_senders_before",

        "fan_in_feature",
        "fan_out_feature",

        "sender_out_degree_before",
        "receiver_in_degree_before",

        "cycle_detected_as_of_time",
        "cycle_count"
    ).summary()
)

In [0]:
# COMMAND ----------

display(
    ml_training_data.select(
        F.sum("high_value_flag")
            .alias("high_value_transactions"),

        F.sum("velocity_flag")
            .alias("velocity_transactions"),

        F.sum("fan_in_flag")
            .alias("fan_in_transactions"),

        F.sum("fan_out_flag")
            .alias("fan_out_transactions"),

        F.sum("cycle_detected_as_of_time")
            .alias("cycle_transactions")
    )
)

In [0]:
# COMMAND ----------

display(
    ml_training_data
    .groupBy("label")
    .agg(
        F.sum("high_value_flag")
            .alias("high_value_count"),

        F.sum("velocity_flag")
            .alias("velocity_count"),

        F.sum("fan_in_flag")
            .alias("fan_in_count"),

        F.sum("fan_out_flag")
            .alias("fan_out_count"),

        F.sum("cycle_detected_as_of_time")
            .alias("cycle_count")
    )
    .orderBy("label")
)

In [0]:
# COMMAND ----------

(
    ml_training_data
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "path",
        ML_TRAINING_PATH
    )
    .saveAsTable(
        ML_TRAINING_TABLE
    )
)

print("ML training dataset saved successfully.")
print("Unity Catalog:", ML_TRAINING_TABLE)
print("S3 location:", ML_TRAINING_PATH)